# LLM Colors Preference Analysis
Analyze the results of adaptive preference elicitation from LLMs on the colors domain, comparing GRUM to the Bradley-Terry baseline.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 1. Load Data
Set the `EXP_DIR` to your experiment output folder.

In [2]:
# Set experiment directories
EXP_DIRS = [
    ROOT / "results" / "llm" / "llm_colors-20260326-114531"
]

results_by_criterion = {}
for exp_dir in EXP_DIRS:
    output_dir = exp_dir / "outputs"
    
    if output_dir.exists():
        for f in sorted(output_dir.glob("*.json")):
            with open(f, "r") as j:
                res = json.load(j)
                crit_base = res.get("criterion", "social")
                
                # Determine embedding label (HS or ST)
                embedding = res.get("agent_config", {}).get("params", {}).get("embedding_method", "")
                if not embedding:
                    # Fallback to filename parsing
                    if "hidden_state" in f.name:
                        emb_label = "HS"
                    elif "sentence_transformer" in f.name:
                        emb_label = "ST"
                    else:
                        emb_label = "unknown"
                else:
                    emb_label = "HS" if "hidden_state" in embedding else "ST"
                
                # Extract model ID if available to distinguish between 0.5B and 0.5B-Instruct
                model_id = res.get("agent_config", {}).get("params", {}).get("model_id", "").split("/")[-1]
                model_label = f" ({model_id})" if model_id else ""
                
                # Create unique key
                crit = f"{crit_base} ({emb_label}){model_label}"
                results_by_criterion[crit] = res

if results_by_criterion:
    print(f"Loaded {len(results_by_criterion)} unique runs from {len(EXP_DIRS)} directories.")
    print(f"Runs: {list(results_by_criterion.keys())}")
else:
    print("No results found! Verify paths in EXP_DIRS.")

## 2.2 The Identifiability Problem: Unbounded Utility Drift

During initial runs without utility normalization, we observed extreme "utility drift". Specifically:
- **ST Embeddings**: Utilities for all alternatives dropped consistently, reaching approximately -40.
- **HS Embeddings**: Utilities became polarized (e.g., all personal weights positive, others negative).

### Theoretical Cause
This occurs because pairwise preference models are only identifiable up to an additive constant. Without an anchoring constraint (like sum-to-zero normalization), the utility values can drift indefinitely. Additionally, if the data lacks **Strong Connectivity** (Condition 1 in the GRUM paper), the solution may be unbounded.

### Visualizing the Drift
Below we load results from the `colors_unbounded` artifact to demonstrate this behavior.

In [ ]:
from grums.experiments.utils import load_experiment_results

# Path to merged unbounded data in artifacts
artifact_path = "/home/lotanamit/grum4llm/artifacts/llm/colors_unbounded"

unbounded_results = {}
for method in ['hs', 'st']:
    path = os.path.join(artifact_path, method)
    if os.path.exists(path):
        unbounded_results[method] = load_experiment_results(path)

if unbounded_results:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    for i, (method, res) in enumerate(unbounded_results.items()):
        # Use a sample run from social criterion
        social_runs = [k for k in res.keys() if 'social' in k]
        if not social_runs: continue
        
        sample_key = social_runs[0]
        history = res[sample_key]['history']
        
        steps = sorted(map(int, history.keys()))
        deltas = [history[str(s)]['grum']['delta'] for s in steps]
        deltas = np.array(deltas)
        
        for j in range(deltas.shape[1]):
            label = color_names[j] if 'color_names' in globals() else f'Color {j}'
            axes[i].plot(steps, deltas[:, j], label=label, color=list(color_palette.values())[j])
        
        axes[i].set_title(f'Unbounded Utility Drift ({method.upper()})')
        axes[i].set_xlabel('Elicitation Step')
        axes[i].set_ylabel('Utility (Delta)')
        axes[i].legend(ncol=2)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 2. Global Preferences: GRUM vs Bradley-Terry
How does the learned intrinsic preference ($\delta$) compare to the social average (BT $\beta$)? 
We expect high correlation if prompt variations are indeed variations around a stable core.

In [3]:
plot_data = []
for crit, res in results_by_criterion.items():
    history = res["history"]
    last_step = str(sorted(map(int, history.keys()))[-1])
    delta = history[last_step]["grum"]["delta"]
    for i, color in enumerate(color_names):
        plot_data.append({"Criterion": crit, "Color": color, "Score": delta[i]})

if plot_data:
    df_plot = pd.DataFrame(plot_data)
    
    plt.figure(figsize=(14, 7))
    # Group by Criterion, Hue by Color
    sns.barplot(data=df_plot, x="Criterion", y="Score", hue="Color", palette=color_palette)
    plt.title("Intrinsic Preferences (Delta) Grouped by Criterion")
    plt.axhline(0, color='black', alpha=0.2)
    plt.legend(title="Color Item")
    plt.show()

### 2.1 Correlation Analysis